# Manual Annotations to CSV Export

This notebook converts your manual analysis JSON file to CSV format for comparison with LLM predictions.

## Instructions:
1. Make sure your manual annotations are saved as `manual_analysis_results.json`
2. Update the configuration below with your search query and gene
3. Run all cells
4. Your CSV will be saved as `{search_query}_manual.csv`

In [ ]:
from pathlib import Path

from fyp25_literature_agents import export_manual_to_csv, setup_logging

# Setup logging
setup_logging()

## Configuration

Update these settings to match your analysis:

In [ ]:
# Your search query and gene
search_term = "PPP2R2A AND cancer"  # Change this to your search query
gene = "PPP2R2A"                    # Change this to your target gene

# Your name/ID (for tracking which annotator)
annotator_name = "student_1"        # Change this to your name or ID

# Input JSON file (your manual annotations)
input_json = Path("manual_analysis_results.json")

# Output CSV file (automatically named)
# Example: "PPP2R2A AND cancer" becomes "ppp2r2a_and_cancer_manual.csv"
output_filename = search_term.lower().replace(" ", "_") + "_manual.csv"
output_csv = Path(output_filename)

print("Configuration:")
print(f"  Search term: {search_term}")
print(f"  Gene: {gene}")
print(f"  Annotator: {annotator_name}")
print(f"  Input: {input_json}")
print(f"  Output: {output_csv}")

## Check Input File

Verify your JSON file exists and is valid:

In [ ]:
import json

if not input_json.exists():
    print(f"❌ ERROR: Input file not found: {input_json}")
    print("\nPlease create your manual annotations using the manual_analysis_template.txt")
    print("Save your annotations as 'manual_analysis_results.json' in this directory.")
else:
    try:
        with open(input_json) as f:
            data = json.load(f)

        if not isinstance(data, list):
            print(f"❌ ERROR: Expected JSON array, got {type(data).__name__}")
        else:
            print(f"✅ Found valid JSON file with {len(data)} articles")

            # Count cancers
            total_cancers = sum(len(article.get("cancers", [])) for article in data)
            print(f"✅ Total cancer classifications: {total_cancers}")

            # Show first article as preview
            if data:
                print("\nPreview of first article:")
                first = data[0]
                print(f"  PMID: {first.get('pmid', 'N/A')}")
                print(f"  Title: {first.get('title', 'N/A')[:80]}...")
                print(f"  Cancers: {len(first.get('cancers', []))}")
                print(f"  Clinical: {first.get('has_clinical', False)}")
                print(f"  Basic: {first.get('has_basic', False)}")
                print(f"  Mutations: {first.get('has_mutations', False)}")

    except json.JSONDecodeError as e:
        print(f"❌ ERROR: Invalid JSON format: {e}")
        print("\nPlease check your JSON syntax. Common issues:")
        print("  - Missing commas between array items")
        print("  - Missing quotes around strings")
        print("  - Trailing commas before closing brackets")
        print("  - Use 'true'/'false' (lowercase) for booleans")

## Export to CSV

Convert your JSON annotations to CSV format:

In [ ]:
if input_json.exists():
    try:
        export_manual_to_csv(
            json_file=input_json,
            output_csv=output_csv,
            search_term=search_term,
            gene=gene,
            annotator_name=annotator_name,
        )

        print(f"✅ SUCCESS! Manual annotations exported to: {output_csv}")
        print("\nYou can now use this CSV file to compare with LLM predictions.")

    except Exception as e:
        print(f"❌ ERROR during export: {e}")
        print("\nPlease check your JSON format matches the template.")
else:
    print("❌ Skipping export - input file not found")

## Preview CSV Output

Display first few rows of the exported CSV:

In [ ]:
import pandas as pd

if output_csv.exists():
    df = pd.read_csv(output_csv)

    print("CSV Summary:")
    print(f"  Total rows: {len(df)}")
    print(f"  Unique articles (PMIDs): {df['pmid'].nunique()}")
    print(f"  Unique cancer types: {df['cancer_type'].nunique()}")
    print("\nRole distribution:")
    print(df["role"].value_counts())
    print("\nConfidence distribution:")
    print(df["confidence"].value_counts())

    print(f"\nFirst 5 rows:")
    display(df.head())
else:
    print("❌ CSV file not found - export may have failed")

## Validate Data Quality

Check for common issues in your annotations:

In [ ]:
if output_csv.exists():
    df = pd.read_csv(output_csv)

    print("Data Quality Checks:")
    print("=" * 50)

    # Check for empty cancer types
    empty_cancers = df["cancer_type"].isna() | (df["cancer_type"] == "")
    if empty_cancers.any():
        print(f"⚠️  Warning: {empty_cancers.sum()} rows with empty cancer_type")
        print("   (This is OK if articles don't discuss cancer)")
    else:
        print("✅ All rows have cancer_type specified")

    # Check for valid roles
    valid_roles = ["tumor_suppressor", "oncogene", "both", "unclear", ""]
    invalid_roles = df[~df["role"].isin(valid_roles)]
    if len(invalid_roles) > 0:
        print(f"⚠️  Warning: {len(invalid_roles)} rows with invalid role")
        print(f"   Invalid values: {invalid_roles['role'].unique().tolist()}")
    else:
        print("✅ All roles are valid")

    # Check for valid confidence
    valid_confidence = ["high", "medium", "low", ""]
    invalid_confidence = df[~df["confidence"].isin(valid_confidence)]
    if len(invalid_confidence) > 0:
        print(f"⚠️  Warning: {len(invalid_confidence)} rows with invalid confidence")
        print(f"   Invalid values: {invalid_confidence['confidence'].unique().tolist()}")
    else:
        print("✅ All confidence levels are valid")

    # Check for duplicate PMIDs with same cancer type
    duplicates = df[df.duplicated(subset=["pmid", "cancer_type"], keep=False)]
    if len(duplicates) > 0:
        print(f"⚠️  Warning: {len(duplicates)} duplicate PMID+cancer_type combinations")
        print("   This usually means you listed the same cancer twice for one article")
        print(f"   Duplicates: {duplicates[['pmid', 'cancer_type', 'role']].to_dict('records')}")
    else:
        print("✅ No duplicate PMID+cancer_type combinations")

    # Check boolean flags
    for flag in ["has_clinical", "has_basic", "has_mutations"]:
        if df[flag].dtype == "bool":
            print(f"✅ {flag}: {df[flag].sum()} True, {(~df[flag]).sum()} False")
        else:
            print(f"⚠️  Warning: {flag} should be boolean, got {df[flag].dtype}")

    print("\n" + "=" * 50)
    print("Data quality check complete!")
else:
    print("❌ CSV file not found")

## Next Steps

Your manual annotations have been exported to CSV! 

### To compare with LLM predictions:

1. **Generate LLM predictions** (if you haven't already):
   - Open `model_comparison.ipynb`
   - Run the full model comparison
   - This creates `model_comparison_results.csv`

2. **Compare manual vs LLM**:
   - Open `../examples/compare_manual_vs_llm.ipynb`
   - Follow the notebook to calculate metrics
   - View precision, recall, F1 scores

### Files created:
- ✅ `manual_analysis_results.json` (your annotations)
- ✅ `{search_query}_manual.csv` (exported CSV)

### Files needed for comparison:
- ⏳ `model_comparison_results.csv` (from model_comparison.ipynb)

### Documentation:
- 📖 `MANUAL_ANNOTATION_GUIDE.md` - How to create annotations
- 📖 `MANUAL_VS_LLM_COMPARISON.md` - Complete comparison workflow
- 📖 `manual_analysis_template.txt` - JSON template and guidelines